In [ ]:
# from tqdm.notebook import tqdm
from tqdm import tqdm

import pickle

import os
from dotenv import load_dotenv

from IPython.display import display, Markdown

import json

from pythonjsonlogger.json import JsonFormatter

from langchain_openai import ChatOpenAI

from adr import adr
import utils

from adr_checking import ADRChecker

import logging
import sys

import logging

In [2]:
# Set of ADRs to check
ENGLISH_DATAPATH = './data/LLM4ADR-adrs__adrs_english.pickle'

print(load_dotenv())

# Configure LLM
# os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
# os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o-mini'
print(os.environ["OPENAI_MODEL_NAME"])

True
gpt-4.1-mini


In [3]:
llm = ChatOpenAI(
      model=os.environ["OPENAI_MODEL_NAME"], temperature=0.0, 
      # callback_manager=CallbackManager([StreamingStdOutCallbackHandler()])
)
# llm.invoke("Tell me a joke about data scientists")

---

In [4]:
# Load all projects and their ADRs
relative_path = os.path.join('..', ENGLISH_DATAPATH)
with open(relative_path,'rb') as file:
    dict_adrs = pickle.load(file)
print(len(dict_adrs), "projects/organizations?")

projects_dict = utils.process_projects(dict_adrs, min_adrs_per_project=5, min_adr_length=500)
valid_projects = projects_dict['valid_projects']

docs = utils.get_documents(valid_projects, dict_adrs, field='raw')
print(len(docs), "ADRs considered within valid projects")

547 projects/organizations?


organizations:   0%|          | 0/547 [00:00<?, ?it/s]

4316 ADRs considered within valid projects


In [5]:
# Example: get all the ADRs for a given organization-project pair
adrs_to_check = utils.get_documents_by_key(('SAP', 'cloud-sdk-js'), dict_adrs, field='raw')
adrs_to_check

{'0001-circuit-breaker-options.md': "# Circuit Breaker Options\n\n_Date_: 2019-07-29\n\n## Context: What Triggered this Discussion?\n\nWe recently decided to increase the default timeout of our circuit breakers to 10 seconds.\nThis is pretty high for fail fast but necessary to prevent some requests from failing.\nThe underlying issue is currently that we cannot simply make timeout into a parameter, because that would mean creating a new circuit breaker which would dismiss all of the state the previous one accumulated.\nMore generally: circuit breakers inherently need state to decide when to e.g. open a circuit.\n\n## Idea: Central Circuit Breaker (Options) Registry\n\nWe know all of the places where we use circuit breakers (currently for the XSUAA and the destination service IIRC).\nSo what we could do is expose some kind of central registry that allows overriding the options for that use case or even to register custom circuit breakers for specific scenarios.\n\n```ts\nregistry.getDes

In [6]:
# Create the checker
checker = ADRChecker(llm)

In [7]:
# Classify 1 single ADR (randomly chosen), including ADR metadata. This check makes several LLM calls internally.
example_adr = list(adrs_to_check.values())[8]
# result = checker.check_madr_adherence(example_adr) #, metadata={'organization': 'SAP', 'project': 'cloud-sdk-js'})
# result = checker.check_sections(example_adr) #, metadata={'organization': 'SAP', 'project': 'cloud-sdk-js'})

result = checker.check(example_adr, metadata={'organization': 'cloud-sdk-js', 'project': 'SAP'})
print(json.dumps(result, indent=4))

display(Markdown(example_adr))

{"asctime": "2026-01-06 13:21:07,422", "levelname": "INFO", "name": "adr_checking", "message": "Classifying ADR with 298 tokens."}
{"asctime": "2026-01-06 13:21:24,771", "levelname": "INFO", "name": "adr_checking", "message": "Classifying ADR with 298 tokens."}
{
    "section_assessments": [
        {
            "section_name": "Context",
            "presence": "No",
            "content_quality": "No",
            "purpose_consistency": "No",
            "justification": "The ADR does not include a section titled 'Context'. The content that might fulfill the role of 'Context' is instead under the heading 'Motivation'. However, the 'Motivation' section contains detailed explanation and reasoning about the problem and the decision to be made, which overlaps with the intended purpose of 'Context' and 'Considered Options'. The content is project-specific and meaningful but is misplaced under 'Motivation' rather than 'Context'. Therefore, presence is 'No', alternate_title is ['Motivation

## Motivation

Typescript allows for variadic functions e.g.:

```
function doSomething(...strings:string[]){
}
```

You can call the method with no, one, two, ... arguments.
We used this feature quite regularly for example for the `and` and `or` filter functions.
In the TypeScript universe there is also nothing wrong wit it, because you will get a type error if you call the function with an array.

```
function doSomething(['a','b','c']) //type error
```

However, in the JavaScript use case you do not get a type error if you call it with an array only a strange error at runtime.
From the method signature it also looks like an array is a valid input.
Hence, we decided to be lenient to the users and allow also for:

```
doSomething([])
doSomething([a])
doSomething([a,b])
```

in addition to the already possible:

```
doSomething()
doSomething('a')
doSomething('a','b')
```

## Solution

We use method overloading:

```
function functionWithVariableArguments(...varargs: string[]);
function functionWithVariableArguments(array: string[]);
function functionWithVariableArguments(
firstOrArray: undefined | string | string[],
...rest: string[]
): string[] {

}
```

and created a little helper method `variableArgumentToArray()` to merge the two argument `first` and `rest` to an array.


---

## Applying the Checker to the Dataset of ADRs

In [8]:
ALL_PROJECTS_RESULTS = './results/all_projects-checks_results.json'

In [ ]:
# logging.disable(logging.INFO) 
logging.getLogger('adr_checking').setLevel(level=logging.CRITICAL)

relative_path = os.path.join('..', ALL_PROJECTS_RESULTS)

# Now use batch mode to check all the ADRs for all the projects
all_results = []
if not os.path.isfile(relative_path):
    print(f"Checking results for all (valid) {len(valid_projects)} projects")
    for org, project in tqdm(valid_projects, desc="Processing org/projects"):
        adrs_to_check = utils.get_documents_by_key((org, project), dict_adrs, field='raw')
        # print("==", org, project, ":", len(adrs_to_check), "ADRs to check")
        results = checker.check_batch(adrs_to_check, parallel=True, project=project, organization=org, as_dict=True)
        all_results.extend(results)

# logging.basicConfig(level=logging.INFO)
# logging.disable(logging.NOTSET)
    print()
    print("Saving JSON results to", relative_path)
    ADRChecker.save_results(all_results, relative_path)

print("End.")

Checking results for all (valid) 312 projects


Processing org/projects:   4%|▍         | 13/312 [12:08<4:05:07, 49.19s/it]

In [ ]:
print(len(all_results),"ADRs processed.")

---